### Webscraping Toyota Valenzuela PMS Costs

To calculate a defensible Average Order Value (AOV) for Toyota Valenzuela’s after-sales operations, a custom Python script was developed to scrape and extract unstructured pricing data from local automotive discussion forums (e.g., Reddit’s r/Gulong).

Because automotive text data is highly unstructured and prone to abbreviations (where "k" can mean either thousands of pesos or kilometers), a multi-layered Natural Language Processing (NLP) and Regular Expression (Regex) pipeline was built to isolate valid financial data from mechanical metrics.

In [2]:
import json
import pandas as pd
import os
import glob

In [4]:
file_path = 'PMS TOYOTA/*.json'

In [5]:
extracted_text = []

In [6]:
file_list = glob.glob(file_path)

for file_path in file_list:
    file_name = os.path.basename(file_path)
    with open(file_path, 'r', encoding='utf-8') as file:
        raw_json = json.load(file)

        post_data = raw_json[0]['data']['children'][0]['data']
        title = post_data.get('title', '')
        body = post_data.get('selftext', '')

        extracted_text.append({
            'Source_file': file_name,
            'Type': 'Main Post',
            'Content': f"{title}\n{body}"
        })

        comments = raw_json[1]['data']['children']
            
        for comment in comments:
            comment_data = comment['data']
            if 'body' in comment_data:
                extracted_text.append({
                    'Source_File': file_name,
                    'Type': 'Comment',
                    'Content': comment_data['body']
                })

In [7]:
df =  pd.DataFrame(extracted_text)

In [10]:
print(df.head())

  Source_file       Type                                            Content  \
0  PMS 1.json  Main Post  PMS for 40k odo.tama lang ba yung ganito price?\n   
1         NaN    Comment  mukhang hindi tama.  not sure kung ganyan na p...   
2         NaN    Comment  Most of these look fair-ish in price. Your pro...   
3         NaN    Comment   Kung wala na sa warranty mag 3rd party ka nalang   
4         NaN    Comment                      Anong sasakyan to? Ang mahal.   

  Source_File  
0         NaN  
1  PMS 1.json  
2  PMS 1.json  
3  PMS 1.json  
4  PMS 1.json  


### Median Average Order Value

In [13]:
import re
import numpy as np

In [15]:
pattern = r'((?:php|₱)\s*\d{1,3}(?:,\d{3})*|\b\d{1,2}(?:\.\d+)?k(?!\s*(?:km|kms|pms|service)))'

df['Raw_Price'] = df['Content'].str.extract(pattern, flags=re.IGNORECASE)[0]

In [16]:
df['Cleaned_Text'] = df['Content'].str.replace(r'\b\d{1,3}k\s*(?:km|kms|pms|mileage)\b', ' [ODOMETER] ', flags=re.IGNORECASE, regex=True)

In [18]:
df['Raw_Price'] = df['Cleaned_Text'].str.extract(pattern, flags=re.IGNORECASE)[0]

In [19]:
def clean_price(price_str):
    if pd.isna(price_str):
        return np.nan
    
    price_str = str(price_str).lower().replace(',', '').strip()
    
    if 'k' in price_str:
        num_part = re.sub(r'[^\d.]', '', price_str)
        if num_part:
            return float(num_part) * 1000
        return np.nan
    else:
        num_part = re.sub(r'[^\d.]', '', price_str)
        if num_part:
            return float(num_part)
        return np.nan

In [20]:
df['Clean_Price'] = df['Raw_Price'].apply(clean_price)

In [21]:
valid_prices = df[(df['Clean_Price'] >= 3000) & (df['Clean_Price'] <= 50000)]['Clean_Price']

In [27]:
valid_prices.describe

<bound method NDFrame.describe of 0     40000.0
16    40000.0
22    20000.0
23    35000.0
25    45000.0
26     5000.0
28     6000.0
29    30000.0
31    40000.0
32    10000.0
34    10000.0
36    10000.0
37     7000.0
38    10000.0
49    20000.0
Name: Clean_Price, dtype: float64>

In [31]:
valid_prices.mean()


np.float64(21866.666666666668)

In [32]:
valid_prices.median()

np.float64(20000.0)